In [ ]:
from pathlib import Path
import json,sys
destination=Path('/kaggle/working/resolution_analysis_code');destination.mkdir(exist_ok=True)
sources={'select_resolution.py': '"""Choose the prespecified resolution/architecture using development predictions only."""\nimport argparse\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.metrics import roc_auc_score\n\n\ndef select(low, high):\n    candidates=[]; reference=None\n    for size,root in [(224,Path(low)),(320,Path(high))]:\n        config=json.loads((root/\'inference_config.json\').read_text())\n        if config.get(\'image_size\',224)!=size:\n            raise ValueError(\'Resolution does not match supplied arm\')\n        manifest=pd.read_csv(root/\'split_manifest.csv\')\n        development=manifest[manifest.split!=\'test\'][[\'sha256\',\'patient\',\'label\',\'split\']].sort_values(\'sha256\').reset_index(drop=True)\n        if reference is None: reference=development\n        elif not development.equals(reference):\n            raise ValueError(\'Development cohorts differ between resolution arms\')\n        expected=development[development.split==\'selection\'].sort_values(\'sha256\').reset_index(drop=True)\n        if expected.sha256.duplicated().any() or set(expected.label)!={0,1}:\n            raise ValueError(\'Invalid selection cohort\')\n        means={}\n        for architecture in [\'efficientnet_b0\',\'densenet121\',\'resnet18\']:\n            values=[]\n            for seed in [42,123,2026]:\n                trial=f\'{architecture}_seed_{seed}\'\n                frame=pd.read_csv(root/trial/\'selection_predictions.csv\').sort_values(\'sha256\').reset_index(drop=True)\n                for field in [\'sha256\',\'patient\']:\n                    if not frame[field].equals(expected[field]):\n                        raise ValueError(\'Selection identity mismatch: \'+trial)\n                if not np.array_equal(frame.y_true,expected.label):\n                    raise ValueError(\'Selection label mismatch: \'+trial)\n                p=frame.probability.to_numpy(float)\n                if not np.isfinite(p).all() or not ((p>=0)&(p<=1)).all():\n                    raise ValueError(\'Invalid selection scores\')\n                values.append(float(roc_auc_score(frame.y_true,p)))\n            mean=float(np.mean(values)); means[architecture]=mean\n            candidates.append(dict(image_size=size,architecture=architecture,\n                                   selection_auroc_by_seed=values,mean_selection_auroc=mean))\n        chosen=sorted(means,key=lambda name:(-means[name],name))[0]\n        if config[\'architecture\']!=chosen:\n            raise ValueError(\'Arm export does not follow architecture selection rule\')\n    ordered=sorted(candidates,key=lambda c:(-c[\'mean_selection_auroc\'],c[\'image_size\'],c[\'architecture\']))\n    return dict(selected=ordered[0],candidates=ordered,\n                criterion=\'Highest mean model-selection AUROC across seeds; ties prefer 224 then alphabetical architecture\',\n                test_predictions_accessed=False,\n                interpretation=\'Exploratory protocol extension; expanded selection search can increase optimism. Both arms must still be reported.\')\n\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'low\',type=Path); parser.add_argument(\'high\',type=Path)\n    parser.add_argument(\'--output\',type=Path,required=True)\n    args=parser.parse_args(); result=select(args.low,args.high)\n    args.output.parent.mkdir(parents=True,exist_ok=True)\n    args.output.write_text(json.dumps(result,indent=2))\n    print(json.dumps(result,indent=2))\n', 'inspect_selection_stability.py': '"""Descriptive development-group bootstrap of fixed architecture/resolution scores."""\nimport argparse\nfrom collections import Counter\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.metrics import roc_auc_score\n\nfrom select_resolution import select\n\n\ndef inspect(low,high,replicates=1000):\n    if replicates<1: raise ValueError(\'replicates must be positive\')\n    decision=select(low,high)  # Validates both development cohorts and every member prediction.\n    candidates=sorted(decision[\'candidates\'],key=lambda c:(c[\'image_size\'],c[\'architecture\']))\n    names=[f"{c[\'image_size\']}:{c[\'architecture\']}" for c in candidates]\n    arrays=[];reference=None\n    for candidate in candidates:\n        root=Path(low if candidate[\'image_size\']==224 else high)\n        members=[]\n        for seed in [42,123,2026]:\n            frame=pd.read_csv(root/f"{candidate[\'architecture\']}_seed_{seed}"/\'selection_predictions.csv\').sort_values(\'sha256\').reset_index(drop=True)\n            if reference is None: reference=frame\n            members.append(frame.probability.to_numpy(float))\n        arrays.append(members)\n    scores=np.asarray(arrays);y=reference.y_true.to_numpy(int)\n    groups=[np.flatnonzero(reference.patient.to_numpy()==g) for g in reference.patient.unique()]\n    rng=np.random.default_rng(42);counts=Counter();sample_scores=[];ties=0\n    for _ in range(replicates):\n        indices=np.concatenate([groups[i] for i in rng.integers(0,len(groups),len(groups))])\n        labels=y[indices]\n        if len(np.unique(labels))!=2: continue\n        means=np.array([np.mean([roc_auc_score(labels,member[indices]) for member in members]) for members in scores])\n        winner=int(np.argmax(means))  # Candidate order implements the frozen tie rule.\n        counts[names[winner]]+=1\n        ties+=int(np.count_nonzero(means==means.max())>1)\n        sample_scores.append(means)\n    if not sample_scores: raise ValueError(\'No usable two-class bootstrap samples\')\n    sample_scores=np.asarray(sample_scores);used=len(sample_scores)\n    original=decision[\'selected\'];chosen=f"{original[\'image_size\']}:{original[\'architecture\']}"\n    return dict(status=\'Exploratory development-sample sensitivity analysis\',\n        images=len(y),filename_proxy_groups=len(groups),requested_replicates=replicates,\n        usable_replicates=used,replicates_with_tied_maximum=ties,\n        original_development_choice=chosen,\n        winner_counts={name:counts[name] for name in names},\n        winner_fractions={name:counts[name]/used for name in names},\n        mean_selection_auroc_percentile_95={name:np.percentile(sample_scores[:,i],[2.5,97.5]).tolist() for i,name in enumerate(names)},\n        test_predictions_accessed=False,changes_to_deployed_choice=False,\n        interpretation=\'Resamples the same selection groups jointly for all fixed models. Winner frequencies are descriptive stability measures, not posterior probabilities, superiority probabilities or external performance estimates. Checkpoints were selected using this partition, so selection optimism remains. Training and checkpoint selection are not repeated; three fixed seeds are averaged without seed resampling. The frozen original choice is retained.\')\n\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'low\',type=Path);parser.add_argument(\'high\',type=Path)\n    parser.add_argument(\'--output\',type=Path,required=True)\n    parser.add_argument(\'--replicates\',type=int,default=1000)\n    args=parser.parse_args();report=inspect(args.low,args.high,args.replicates)\n    args.output.parent.mkdir(parents=True,exist_ok=True)\n    args.output.write_text(json.dumps(report,indent=2))\n    print(json.dumps(report,indent=2))\n', 'compare_experiments.py': '"""Paired descriptive comparison of frozen predictions; never fits thresholds."""\nimport argparse\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.metrics import roc_auc_score, average_precision_score\n\n\ndef metrics(frame):\n    y = frame.y_true.to_numpy(dtype=int)\n    p = frame.probability.to_numpy(dtype=float)\n    pred = frame.prediction.to_numpy(dtype=int)\n    return {\n        \'auroc\': float(roc_auc_score(y, p)),\n        \'average_precision\': float(average_precision_score(y, p)),\n        \'accuracy\': float(np.mean(y == pred)),\n        \'sensitivity\': float(np.mean(pred[y == 1] == 1)),\n        \'specificity\': float(np.mean(pred[y == 0] == 0)),\n        \'brier_score\': float(np.mean((y - p) ** 2)),\n    }\n\n\ndef compare(baseline_dir, followup_dir, replicates=1000):\n    if replicates < 1:\n        raise ValueError(\'replicates must be positive\')\n    frames = []\n    for directory in (baseline_dir, followup_dir):\n        frame = pd.read_csv(Path(directory) / \'test_predictions.csv\')\n        required = {\'sha256\', \'patient\', \'y_true\', \'probability\', \'prediction\'}\n        if not required.issubset(frame.columns) or frame.sha256.duplicated().any():\n            raise ValueError(\'Missing fields or duplicate image hashes\')\n        if set(frame.y_true) != {0, 1} or not set(frame.prediction).issubset({0, 1}):\n            raise ValueError(\'Expected binary labels with both classes represented\')\n        if not np.isfinite(frame.probability).all() or not frame.probability.between(0, 1).all():\n            raise ValueError(\'Invalid probabilities\')\n        if frame.patient.isna().any() or frame.sha256.isna().any():\n            raise ValueError(\'Missing pairing or cluster identifiers\')\n        frames.append(frame.sort_values(\'sha256\').reset_index(drop=True))\n    a, b = frames\n    for column in (\'sha256\', \'patient\', \'y_true\'):\n        if not a[column].equals(b[column]):\n            raise ValueError(\'Test cohorts or labels differ: \' + column)\n    ma, mb = metrics(a), metrics(b)\n    groups = [np.flatnonzero(a.patient.to_numpy() == group) for group in a.patient.unique()]\n    rng = np.random.default_rng(42)\n    differences = []\n    for _ in range(replicates):\n        idx = np.concatenate([groups[i] for i in rng.integers(0, len(groups), len(groups))])\n        if a.iloc[idx].y_true.nunique() != 2:\n            continue\n        sa, sb = metrics(a.iloc[idx]), metrics(b.iloc[idx])\n        differences.append({key: sb[key] - sa[key] for key in sa})\n    if not differences:\n        raise ValueError(\'No bootstrap sample contained both classes\')\n    return {\n        \'status\': \'Exploratory paired comparison; original test results already inspected\',\n        \'direction\': \'followup minus baseline; lower is better for Brier score only\',\n        \'images\': len(a), \'filename_proxy_groups\': len(groups),\n        \'baseline\': ma, \'followup\': mb,\n        \'difference\': {key: mb[key] - ma[key] for key in ma},\n        \'paired_cluster_bootstrap_95ci\': {\n            key: np.percentile([d[key] for d in differences], [2.5, 97.5]).tolist()\n            for key in ma},\n        \'bootstrap_replicates\': len(differences),\n        \'limitations\': \'Filename groups are proxies. Intervals describe test sampling, not training variation, model selection uncertainty or external generalization. No multiplicity adjustment. Thresholds remain those saved by each experiment.\'\n    }\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'baseline\', type=Path)\n    parser.add_argument(\'followup\', type=Path)\n    parser.add_argument(\'--output\', type=Path, required=True)\n    parser.add_argument(\'--replicates\', type=int, default=1000)\n    args = parser.parse_args()\n    report = compare(args.baseline, args.followup, args.replicates)\n    args.output.parent.mkdir(parents=True, exist_ok=True)\n    args.output.write_text(json.dumps(report, indent=2))\n    print(json.dumps(report, indent=2))\n', 'analyze_benchmark_comparison.py': '"""Descriptive paired uncertainty for already-frozen source test predictions."""\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nfrom compare_experiments import compare, metrics\n\n\ndef analyze(previous, benchmark, output, replicates=1000):\n    output=Path(output);output.mkdir(parents=True,exist_ok=True)\n    comparison=compare(previous,benchmark,replicates)\n    frame=pd.read_csv(Path(benchmark)/\'test_predictions.csv\').sort_values(\'sha256\').reset_index(drop=True)\n    groups=[np.flatnonzero(frame.patient.to_numpy()==g) for g in frame.patient.unique()]\n    rng=np.random.default_rng(42); samples=[]\n    for _ in range(replicates):\n        indices=np.concatenate([groups[i] for i in rng.integers(0,len(groups),len(groups))])\n        sample=frame.iloc[indices]\n        if sample.y_true.nunique()==2: samples.append(metrics(sample))\n    if not samples: raise ValueError(\'No usable bootstrap samples\')\n    intervals={name:np.percentile([m[name] for m in samples],[2.5,97.5]).tolist() for name in samples[0]}\n    report={\'status\':\'Exploratory; the original source test was previously inspected\',\n            \'benchmark_metrics\':metrics(frame),\'benchmark_cluster_bootstrap_95ci\':intervals,\n            \'bootstrap_replicates\':len(samples),\'paired_comparison\':comparison,\n            \'interpretation\':\'Conditional test-sampling uncertainty only. Does not capture training, model selection, proxy-group error or external generalization. No threshold refitting or model selection occurs.\'}\n    (output/\'comparison.json\').write_text(json.dumps(report,indent=2))\n    print(\'COMPARISON_DONE\',json.dumps(report),flush=True)\n    return report\n\n\nif __name__==\'__main__\':\n    import argparse\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'previous\',type=Path);parser.add_argument(\'benchmark\',type=Path)\n    parser.add_argument(\'--output\',type=Path,required=True)\n    args=parser.parse_args();analyze(args.previous,args.benchmark,args.output)\n'}
for name,text in sources.items(): (destination/name).write_text(text)
sys.path.insert(0,str(destination))
from select_resolution import select
from inspect_selection_stability import inspect
from analyze_benchmark_comparison import analyze
root=Path('/kaggle/input')
def find_output(name):
    matches=[p.parent for p in root.rglob('inference_config.json') if p.parent.name==name]
    if len(matches)!=1: raise ValueError('Attach exactly one completed output: '+name)
    return matches[0]
low=find_output('pneumonia_benchmark_outputs')
high=find_output('pneumonia_resolution320_outputs')
out=Path('/kaggle/working/resolution_comparison_outputs');out.mkdir(exist_ok=True)
# Save the development-only decision before reading test predictions.
decision=select(low,high)
(out/'development_selection.json').write_text(json.dumps(decision,indent=2))
print('DEVELOPMENT_SELECTION',json.dumps(decision),flush=True)
stability=inspect(low,high)
(out/'selection_stability.json').write_text(json.dumps(stability,indent=2))
print('SELECTION_STABILITY',json.dumps(stability),flush=True)
report=analyze(low,high,out)
report['experiment_mapping']={'baseline':'224-pixel development-selected ensemble','followup':'320-pixel development-selected ensemble'}
(out/'comparison.json').write_text(json.dumps(report,indent=2))
print('RESOLUTION_COMPARISON_DONE',json.dumps(report),flush=True)
